# Mini-Project 4 — A LangGraph Alert-Triage Agent
### IT7075: Applied AI for Cybersecurity · LangGraph

A chain runs straight through. An **agent decides** — and the decision is the interesting part,
because a triage agent can be wrong in two very different ways:

- it **misses** a real incident (the dangerous error), or
- it **escalates noise** and buries your analysts (the expensive error).

You will build the Lecture 3 triage graph, run it against **12 labelled alerts**, and measure both
kinds of error. Then you tune the rule and watch one error trade against the other.

**This project is mostly analysis, not coding.** There are five TODOs and they total **ten lines** —
the graph is small on purpose. Most of the credit is for what you write about the results.

| | LangGraph concept | You write |
|---|---|---|
| **TODO 1** | the **state** (a `TypedDict`) | 2 lines |
| **TODO 2** | a **node** (plain function → dict) | 2 lines |
| **TODO 3** | the **router** for a conditional edge | 1 line |
| **TODO 4** | a node that calls a **tool** | 2 lines |
| **TODO 5** | **build and compile** the graph | 3 lines |

Find them by searching for `###################`:

```
###################  TODO n — title  ###################
# what to do, and how
################
```

> **Runs fully offline.** Every node is ordinary Python — no API key, no model download, no GPU.
> That is deliberate: the point is the *routing decision*, and a deterministic agent lets you
> measure it exactly.

## Part 1 — The state  ·  **TODO 1**

The state is the graph's memory: one dictionary that every node reads from and writes to. Declaring
it as a `TypedDict` is how you say *these* are the fields an alert carries as it moves through the
graph.

This is the first real difference from the LangChain module. There, data flowed along a pipe from
one step to the next. Here every node sees the **whole** state, and a node only returns the keys it
changed.

> **Expected output:** `state fields: ['alert', 'risk', 'cve_info', 'report']`.

In [86]:
import json, os
from typing import TypedDict

ALERTS_FILE = "alerts.json" if os.path.exists("alerts.json") else "../alerts.json"


###################  TODO 1 — declare the state  ###################
# WHAT: add the two missing fields to TriageState.
#
# The agent needs four things in its state, and two are already there:
#     alert     : str   the incoming alert text            (given)
#     risk      : str   "high" or "low", set by TODO 2     <- you add this
#     cve_info  : str   what the tool found, set by TODO 4 <- you add this
#     report    : str   the final summary                  (given)
#
# HOW (the Lecture 2 "Define the State" cell): inside the class, one line per
# field, in the form   name: type
#
# WHY a TypedDict and not a plain dict: it documents the agent's memory in one
# place, so when a node returns {"risk": "high"} you can see where that lands.
################

class TriageState(TypedDict):
    alert: str
    risk: str
    cve_info: str
    report: str


print("state fields:", list(TriageState.__annotations__))

state fields: ['alert', 'risk', 'cve_info', 'report']


## Part 2 — The classify node  ·  **TODO 2**

A node is just a Python function: it takes the state and returns a dict of the keys it changed.

This particular node is the one your whole experiment turns on. It decides `risk` by looking for
words in the alert text — the same rule as the Lecture 3 notebook. Keep `KEYWORDS` as a variable:
Part 7 changes it and re-measures.

> **Expected output:** `high` for the RCE alert, `low` for the failed-login alert.

In [87]:
# The Lecture 3 keyword list. Part 7 will change this and re-measure.
KEYWORDS = ["rce", "ransomware", "exfil", "critical", "admin"]


###################  TODO 2 — write the classify node  ###################
# WHAT: finish classify() so it sets `risk` from the alert text.
#
# HOW (the Lecture 3 "Conditional Edges" cell):
#   * is any keyword present?  Search the LOWERCASED alert text:
#         any(word in state["alert"].lower() for word in KEYWORDS)
#   * a node returns a DICT of just the fields it changed -- here:
#         {"risk": "high"}  or  {"risk": "low"}
#     Do NOT return the whole state, and do not mutate it in place.
#
# NOTE: matching is substring-based, so "admin" also fires on "administrator".
#       Keep that in mind when you read your results -- it matters later.
################

def classify(state: TriageState) -> dict:
    hit = any(word in state["alert"].lower() for word in KEYWORDS)
    return {"risk": "high" if hit else "low"}


# ---- given: try it on two alerts ----------------------------------------
for text in ["Critical RCE exploit attempt against the Apache server",
             "Single failed login from a known corporate device"]:
    print("%-58s -> %s" % (text[:56], classify({"alert": text})["risk"]))

Critical RCE exploit attempt against the Apache server     -> high
Single failed login from a known corporate device          -> low


## Part 3 — The router  ·  **TODO 3**

This is what makes it a graph rather than a chain. A **router** is a function that looks at the
state and returns a *label* — the name of the branch to take next. LangGraph calls it after the
`classify` node and jumps to whichever node that label points at.

The router decides nothing else: no work, no side effects, just "which way".

> **Expected output:** `high -> investigate`, `low -> quick_close`.

In [88]:
###################  TODO 3 — write the router  ###################
# WHAT: return the LABEL of the next branch, based on state["risk"].
#
#     risk == "high"  ->  "investigate"
#     anything else   ->  "quick_close"
#
# HOW (the Lecture 3 route_by_risk function): one line, e.g.
#     return "investigate" if <condition> else "quick_close"
#
# These two strings are labels, not node names -- TODO 5 maps them to real
# nodes. Keeping them separate is what lets you rewire a branch without
# touching this function.
################

def route_by_risk(state: TriageState) -> str:
    return "investigate" if state["risk"] == "high" else "quick_close"


# ---- given ---------------------------------------------------------------
for r in ["high", "low"]:
    print("%-5s -> %s" % (r, route_by_risk({"risk": r})))

high  -> investigate
low   -> quick_close


## Part 4 — A node that calls a tool  ·  **TODO 4**

A **tool** is just a function the agent can call for information it does not have. `lookup_cve` is
given — a tiny stand-in for a threat-intel API. Your job is the node that uses it and writes the
result into the state.

> **Expected output:** the RCE alert returns the `CVE-2026-0001` line; an unmatched alert returns
> the "No specific CVE found" line.

In [89]:
def lookup_cve(keyword: str) -> str:
    """GIVEN: the tool. A stand-in for a threat-intel lookup."""
    db = {"rce": "CVE-2026-0001: critical RCE; patch immediately.",
          "ransomware": "Multiple CVEs; isolate the host and restore from backup.",
          "credential": "Credential theft; force password resets and review logins.",
          "outbound": "Possible data theft; check egress logs and block the destination."}
    for key, value in db.items():
        if key in keyword.lower():
            return value
    return "No specific CVE found; follow standard triage."


###################  TODO 4 — write the investigate node  ###################
# WHAT: call the tool, then return BOTH fields it fills in.
#
# HOW:
#   * info = lookup_cve(state["alert"])
#   * return a dict with two keys:
#         "cve_info" -> info
#         "report"   -> "ESCALATE. " + info
#
# This is the whole idea of a tool: the node does not guess what the CVE is,
# it asks something that knows. Note the node returns two keys at once --
# a node may update as many state fields as it likes.
################

def investigate(state: TriageState) -> dict:
    info = lookup_cve(state["alert"])
    return {"cve_info": info, "report": "ESCALATE. " + info}


def quick_close(state: TriageState) -> dict:
    """GIVEN: the other branch. No tool call, no escalation."""
    return {"cve_info": "", "report": "CLOSE. Logged for monitoring; no action needed."}


# ---- given ---------------------------------------------------------------
for text in ["Critical RCE exploit attempt", "Backup job completed with a warning"]:
    print("%-32s -> %s" % (text, investigate({"alert": text})["cve_info"]))

Critical RCE exploit attempt     -> CVE-2026-0001: critical RCE; patch immediately.
Backup job completed with a warning -> No specific CVE found; follow standard triage.


## Part 5 — Build and compile the graph  ·  **TODO 5**

Now wire the pieces together. The nodes are added for you; you supply the three lines that make it
a runnable graph — where it starts, how the branch works, and the compile step.

```
        classify
           |  route_by_risk
     +-----+-----+
     |           |
investigate   quick_close
     |           |
     +-----+-----+
           END
```

> **Expected output:** the RCE alert reports `ESCALATE. CVE-2026-0001 ...`; the failed-login alert
> reports `CLOSE. Logged for monitoring ...`.

In [90]:
from langgraph.graph import StateGraph, END


#def build_triage_graph():
 #   """Assemble the graph. Called again in Part 7 after KEYWORDS changes."""
  #  builder = StateGraph(TriageState)
   # builder.add_node("classify", classify)
    #builder.add_node("investigate", investigate)
    #builder.add_node("quick_close", quick_close)

    ###################  TODO 5 — wire it up  ###################
    # WHAT: three lines, all from the Lecture 3 "Wire the Branching Graph" cell.
    #
    #   1. entry point -- the graph starts at the "classify" node:
    #          builder.set_entry_point("classify")
    #
    #   2. the conditional edge -- after "classify", call your router and map each
    #      LABEL it can return to the node that should run next:
    #          builder.add_conditional_edges("classify", route_by_risk,
    #                                        {"investigate": "investigate",
    #                                         "quick_close": "quick_close"})
    #      (left side = the label from TODO 3, right side = the node name)
    #
    #   3. compile it into a runnable graph and return it:
    #          return builder.compile()
    #
    # The two edges to END are already there: both branches finish the run.
    ################

    
def build_triage_graph():
    """Assemble the alert triage graph."""

    builder = StateGraph(TriageState)

    builder.add_node("classify", classify)
    builder.add_node("investigate", investigate)
    builder.add_node("quick_close", quick_close)

    # TODO 5: Set the starting node
    builder.set_entry_point("classify")

    # TODO 5: Choose the next node based on risk
    builder.add_conditional_edges(
        "classify",
        route_by_risk,
        {
            "investigate": "investigate",
            "quick_close": "quick_close"
        }
    )

    # Both branches finish at END
    builder.add_edge("investigate", END)
    builder.add_edge("quick_close", END)

    # TODO 5: Compile the graph
    return builder.compile()



graph = build_triage_graph()


# ---- given: a blank state, then one run per alert ------------------------
def run(graph, alert_text):
    """GIVEN: invoke the graph on one alert and hand back the finished state."""
    return graph.invoke({"alert": alert_text, "risk": "", "cve_info": "", "report": ""})


for text in ["Critical RCE exploit attempt against the Apache server",
             "Single failed login from a known corporate device"]:
    out = run(graph, text)
    print("ALERT :", text)
    print("  risk:", out["risk"], "| report:", out["report"], "\n")

ALERT : Critical RCE exploit attempt against the Apache server
  risk: high | report: ESCALATE. CVE-2026-0001: critical RCE; patch immediately. 

ALERT : Single failed login from a known corporate device
  risk: low | report: CLOSE. Logged for monitoring; no action needed. 



In [91]:
import sys
print("Python:", sys.executable)

import langgraph
print("LangGraph imported successfully!")

Python: /Users/rozangsoh/Desktop/MSIT/1st Sem/Applied AI Cybersec/LangGraph_Project/.venv/bin/python
LangGraph imported successfully!


## Part 6 — Score the agent  *(no TODO — run it and read it)*

The coding is done. Everything from here is measurement and interpretation.

`alerts.json` holds **12 real-looking alerts**, each labelled with what a SOC analyst would
actually do: **6 `escalate`** and **6 `close`**. Comparing your agent's route to that label gives
three numbers — and the two errors are not equally bad:

| | Meaning | Cost |
|---|---|---|
| **caught** | escalated something that deserved it | what you want |
| **missed** | closed a real incident | an attacker keeps going, unnoticed |
| **over-escalated** | escalated routine noise | analyst time, and eventually alert fatigue |

> **Expected output with the Lecture 3 keyword list: caught 3/6, missed 3/6, over-escalated 2/6.**
> The agent you just built misses **half** of the real incidents. Part 7 asks what to do about it.

In [92]:
ALERTS = json.load(open(ALERTS_FILE))["alerts"]
print("loaded", len(ALERTS), "labelled alerts\n")


def score(graph, alerts):
    """GIVEN: run every alert and compare the route with the analyst's label."""
    caught = missed = over = 0
    rows = []
    for a in alerts:
        out = run(graph, a["alert"])
        route = "escalate" if out["risk"] == "high" else "close"
        if a["expected"] == "escalate":
            verdict = "caught" if route == "escalate" else "MISSED"
        else:
            verdict = "over-escalated" if route == "escalate" else "closed ok"
        caught += verdict == "caught"
        missed += verdict == "MISSED"
        over += verdict == "over-escalated"
        rows.append({"id": a["id"], "expected": a["expected"], "agent": route,
                     "verdict": verdict, "alert": a["alert"][:52]})
    return caught, missed, over, rows


caught, missed, over, rows = score(graph, ALERTS)
n_esc = sum(a["expected"] == "escalate" for a in ALERTS)
n_close = len(ALERTS) - n_esc

print("caught          %d/%d real incidents" % (caught, n_esc))
print("MISSED          %d/%d real incidents      <-- the dangerous error" % (missed, n_esc))
print("over-escalated  %d/%d routine alerts      <-- the expensive error\n" % (over, n_close))

print("%-5s %-9s %-9s %-15s %s" % ("id", "expected", "agent", "verdict", "alert"))
for r in rows:
    print("%-5s %-9s %-9s %-15s %s" % (r["id"], r["expected"], r["agent"], r["verdict"], r["alert"]))

loaded 12 labelled alerts

caught          3/6 real incidents
MISSED          3/6 real incidents      <-- the dangerous error
over-escalated  2/6 routine alerts      <-- the expensive error

id    expected  agent     verdict         alert
A01   escalate  escalate  caught          Critical RCE exploit attempt against the internet-fa
A02   escalate  escalate  caught          Ransomware canary file modified on the finance file 
A03   escalate  close     MISSED          Large outbound data transfer to an unknown host at 0
A04   escalate  escalate  caught          New local administrator account created on the domai
A05   escalate  close     MISSED          Credential dumping tool signature detected in memory
A06   escalate  close     MISSED          Repeated MFA prompts followed by a successful login 
A07   close     escalate  over-escalated  Scheduled vulnerability scan reported critical findi
A08   close     close     closed ok       User reset their own password via the self-service p
A

## Part 7 — Tune the rule and watch the two errors trade  *(no TODO)*

The obvious fix for a missed incident is to add words. Do that four times over and watch what it
costs.

> **Expected output:**
>
> | keyword list | caught | missed | over-escalated |
> |---|---:|---:|---:|
> | narrow (2 words) | 2/6 | **4/6** | 0/6 |
> | **lecture (5 words)** | 3/6 | **3/6** | 2/6 |
> | wider (8 words) | **6/6** | 0/6 | 2/6 |
> | widest (11 words) | 6/6 | 0/6 | **4/6** |
>
> Read the two error columns against each other. Going from narrow to widest drives missed
> detections to zero and doubles the false escalations. There is no row with zeroes in both.
>
> **And look at *which* alerts stay wrong.** A07 and A12 are over-escalated by every list that
> catches all six real incidents. A07 says *critical* — about findings from our own scanner on a
> sandbox VM. A12 says *Administrator* — about an approved patch in a maintenance window. The
> trigger word is right there in both; what makes them harmless is **context the words do not
> carry**. No keyword list can fix that, which is the most important thing this table has to say.

In [93]:
KEYWORD_LISTS = {
    "narrow (2 words)":   ["rce", "ransomware"],
    "lecture (5 words)":  ["rce", "ransomware", "exfil", "critical", "admin"],
    "wider (8 words)":    ["rce", "ransomware", "exfil", "critical", "admin",
                           "outbound", "credential", "mfa"],
    "widest (11 words)":  ["rce", "ransomware", "exfil", "critical", "admin",
                           "outbound", "credential", "mfa", "failed", "scan", "password"],
}

print("%-20s %-8s %-8s %-15s %s" % ("keyword list", "caught", "missed", "over-escalated", "which ones"))
sweep = []
for name, words in KEYWORD_LISTS.items():
    KEYWORDS[:] = words                      # the classify node reads this list
    c, m, o, rr = score(build_triage_graph(), ALERTS)
    bad = [r["id"] for r in rr if r["verdict"] == "MISSED"]
    fat = [r["id"] for r in rr if r["verdict"] == "over-escalated"]
    sweep.append({"list": name, "caught": c, "missed": m, "over": o})
    print("%-20s %-8s %-8s %-15s missed=%-14s over=%s"
          % (name, "%d/%d" % (c, n_esc), "%d/%d" % (m, n_esc), "%d/%d" % (o, n_close),
             ",".join(bad) or "none", ",".join(fat) or "none"))

KEYWORDS[:] = KEYWORD_LISTS["lecture (5 words)"]     # put it back
print("\nSaved to results.csv")
with open("results.csv", "w", encoding="utf-8") as f:
    f.write("keyword_list,caught,missed,over_escalated\n")
    for s in sweep:
        f.write("%s,%d,%d,%d\n" % (s["list"], s["caught"], s["missed"], s["over"]))

keyword list         caught   missed   over-escalated  which ones
narrow (2 words)     2/6      4/6      0/6             missed=A03,A04,A05,A06 over=none
lecture (5 words)    3/6      3/6      2/6             missed=A03,A05,A06    over=A07,A12
wider (8 words)      6/6      0/6      2/6             missed=none           over=A07,A12
widest (11 words)    6/6      0/6      4/6             missed=none           over=A07,A08,A10,A12

Saved to results.csv


## Part 8 — Memory: why the graph beats a chain  *(no TODO)*

One more LangGraph feature from Lecture 3. Compile the **same** graph with a `MemorySaver` and every
run tagged with a `thread_id` is remembered, so an incident can be picked up later — something a
one-shot chain cannot do.

> **Expected output:** the state saved under `incident-42` is still readable after the run, showing
> the alert, the risk, and the report the agent produced.

In [94]:
from langgraph.checkpoint.memory import MemorySaver

builder = StateGraph(TriageState)
builder.add_node("classify", classify)
builder.add_node("investigate", investigate)
builder.add_node("quick_close", quick_close)
builder.set_entry_point("classify")
builder.add_conditional_edges("classify", route_by_risk,
                              {"investigate": "investigate", "quick_close": "quick_close"})
builder.add_edge("investigate", END)
builder.add_edge("quick_close", END)
graph_with_memory = builder.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "incident-42"}}
out = graph_with_memory.invoke(
    {"alert": "Ransomware canary file modified on the finance file share",
     "risk": "", "cve_info": "", "report": ""}, config)
print("run     :", out["report"])

# ...later, or in another cell, the incident is still there:
saved = graph_with_memory.get_state(config).values
print("recalled:", saved["alert"])
print("          risk =", saved["risk"], "| cve_info =", saved["cve_info"][:48])

print("\nA different thread_id starts a fresh incident:")
other = {"configurable": {"thread_id": "incident-99"}}
print("  incident-99 has state yet?", bool(graph_with_memory.get_state(other).values))

run     : ESCALATE. Multiple CVEs; isolate the host and restore from backup.
recalled: Ransomware canary file modified on the finance file share
          risk = high | cve_info = Multiple CVEs; isolate the host and restore from

A different thread_id starts a fresh incident:
  incident-99 has state yet? False


## Part 9: your own alerts, and your own change to the graph

Everything above ran on the provided alerts, and it was practice. A rule tuned on twelve examples
someone else wrote proves very little, so the rest of this notebook is your own.

First write the data. Copy `alerts_template.json` to `my_alerts.json` and write 12 labelled alerts,
6 you would escalate and 6 you would close. Base them on real alert types such as Windows event
logs, EDR detections, cloud audit trails, or IDS signatures, describe them in your own words, and
put any sources in `SOURCES.md`. Include at least two hard cases: one real incident whose text
contains none of your keywords, and one routine alert that does contain one. For every alert, be
able to say why an analyst would escalate or close it.

No code is provided from here on. Add cells below that:

1. Load your alerts and report how many of each label you have.
2. Run the agent on them and score the three outcomes: caught, missed, and over escalated.
3. Change the rule a few times, one change at a time, and record what each change costs on the two
   error columns. Save the table as `results_mine.csv`.
4. Make at least one change to the graph itself, not just the word list. Add a node, add a tool,
   change the router, or add a second decision, then measure the same alerts again and say what it
   bought you.

> When you are finished, run the whole notebook top to bottom and save it with the outputs showing,
> then commit it to your private GitHub repository along with `my_alerts.json` and your results.
> See `MiniProject_Agent.md` section 6.1 for the folder structure.

In [95]:

# PART 9 - Load and score my 12 alerts

import json
from collections import Counter

with open("my_alerts.json", "r", encoding="utf-8") as f:
    MY_ALERTS = json.load(f)["alerts"]

counts = Counter(a["expected"] for a in MY_ALERTS)

print("Total alerts:", len(MY_ALERTS))
print("Escalate:", counts["escalate"])
print("Close:", counts["close"])

assert len(MY_ALERTS) == 12
assert counts["escalate"] == 6
assert counts["close"] == 6
assert len({a["id"] for a in MY_ALERTS}) == 12

# Restore original lecture keywords
KEYWORDS[:] = KEYWORD_LISTS["lecture (5 words)"]

# Score my own alerts
my_caught, my_missed, my_over, my_rows = score(
    build_triage_graph(),
    MY_ALERTS
)

print("\nMY ALERTS - BASELINE")
print("Caught:", my_caught, "/6")
print("Missed:", my_missed, "/6")
print("Over-escalated:", my_over, "/6")

print("\nIndividual decisions:")
for row in my_rows:
    print(
        row["id"],
        "| Expected:", row["expected"],
        "| Agent:", row["agent"],
        "| Verdict:", row["verdict"]
    )


Total alerts: 12
Escalate: 6
Close: 6

MY ALERTS - BASELINE
Caught: 2 /6
Missed: 4 /6
Over-escalated: 3 /6

Individual decisions:
A01 | Expected: escalate | Agent: escalate | Verdict: caught
A02 | Expected: escalate | Agent: escalate | Verdict: caught
A03 | Expected: escalate | Agent: close | Verdict: MISSED
A04 | Expected: escalate | Agent: close | Verdict: MISSED
A05 | Expected: escalate | Agent: close | Verdict: MISSED
A06 | Expected: escalate | Agent: close | Verdict: MISSED
A07 | Expected: close | Agent: escalate | Verdict: over-escalated
A08 | Expected: close | Agent: close | Verdict: closed ok
A09 | Expected: close | Agent: close | Verdict: closed ok
A10 | Expected: close | Agent: close | Verdict: closed ok
A11 | Expected: close | Agent: escalate | Verdict: over-escalated
A12 | Expected: close | Agent: escalate | Verdict: over-escalated


In [96]:

# PART 9 - Step 2: Keyword tuning

MY_RESULTS = []

for name, words in KEYWORD_LISTS.items():
    KEYWORDS[:] = words

    c, m, o, experiment_rows = score(
        build_triage_graph(), MY_ALERTS
    )

    missed_ids = [
        r["id"] for r in experiment_rows
        if r["verdict"] == "MISSED"
    ]

    over_ids = [
        r["id"] for r in experiment_rows
        if r["verdict"] == "over-escalated"
    ]

    MY_RESULTS.append({
        "configuration": name,
        "caught": c,
        "missed": m,
        "over_escalated": o,
        "correctly_closed": 6 - o,
        "missed_ids": ",".join(missed_ids),
        "over_ids": ",".join(over_ids)
    })

    print(
        name,
        "| Caught:", c,
        "| Missed:", m,
        "| Over-escalated:", o
    )

# Restore original lecture keywords
KEYWORDS[:] = KEYWORD_LISTS["lecture (5 words)"]


narrow (2 words) | Caught: 1 | Missed: 5 | Over-escalated: 1
lecture (5 words) | Caught: 2 | Missed: 4 | Over-escalated: 3
wider (8 words) | Caught: 4 | Missed: 2 | Over-escalated: 3
widest (11 words) | Caught: 4 | Missed: 2 | Over-escalated: 5


### Your tuning and your change to the graph

In [97]:
###################  YOUR TUNING AND YOUR GRAPH CHANGE  ###################
# Change the rule one step at a time and record what each change costs.
# Then change the graph itself and measure the same alerts again.
# Save the table as results_mine.csv.
################

In [98]:

# PART 9 - Step 3A: Context lookup tool and node

class ContextTriageState(TriageState):
    context: str


def lookup_alert_context(alert_text: str) -> str:
    """Simulated context lookup using alert descriptions."""
    text = alert_text.lower()

    # Known authorized activity in this lab dataset
    if ("approved maintenance window" in text
            or "approved change ticket" in text):
        return "authorized"

    # Suspicious activity not always detected by keywords
    if ("public read access" in text
            or "disabling endpoint protection" in text):
        return "suspicious"

    return "unknown"


def check_context(state: ContextTriageState) -> dict:
    """Adjust the risk based on additional context."""
    context = lookup_alert_context(state["alert"])

    if context == "suspicious":
        risk = "high"
    elif context == "authorized":
        risk = "low"
    else:
        risk = state["risk"]

    return {
        "context": context,
        "risk": risk
    }


print("Context lookup tool and node created!")


Context lookup tool and node created!


In [99]:

# PART 9 - Step 3B: Build enhanced LangGraph

def build_context_graph():
    builder = StateGraph(ContextTriageState)

    builder.add_node("classify", classify)
    builder.add_node("check_context", check_context)
    builder.add_node("investigate", investigate)
    builder.add_node("quick_close", quick_close)

    builder.set_entry_point("classify")

    # New node between classify and the decision
    builder.add_edge("classify", "check_context")

    builder.add_conditional_edges(
        "check_context",
        route_by_risk,
        {
            "investigate": "investigate",
            "quick_close": "quick_close"
        }
    )

    builder.add_edge("investigate", END)
    builder.add_edge("quick_close", END)

    return builder.compile()


# Use the same eight keywords for comparison
KEYWORDS[:] = KEYWORD_LISTS["wider (8 words)"]

enhanced_graph = build_context_graph()

# Test A03, which the original graph missed
test = run(enhanced_graph, MY_ALERTS[2]["alert"])

print("Alert:", MY_ALERTS[2]["id"])
print("Context:", test["context"])
print("Risk:", test["risk"])
print("Report:", test["report"])


Alert: A03
Context: suspicious
Risk: high
Report: ESCALATE. No specific CVE found; follow standard triage.


In [100]:

# PART 9 - Step 3C: Compare original vs enhanced graph

# Keep keyword settings identical for both graphs
KEYWORDS[:] = KEYWORD_LISTS["wider (8 words)"]

original_graph = build_triage_graph()
enhanced_graph = build_context_graph()

original_c, original_m, original_o, original_rows = score(
    original_graph, MY_ALERTS
)

new_c, new_m, new_o, new_rows = score(
    enhanced_graph, MY_ALERTS
)

print("ORIGINAL GRAPH (8 keywords)")
print("Caught:", original_c)
print("Missed:", original_m)
print("Over-escalated:", original_o)

print("\nENHANCED GRAPH (8 keywords + context)")
print("Caught:", new_c)
print("Missed:", new_m)
print("Over-escalated:", new_o)

print("\nCHANGED DECISIONS:")
for old, new in zip(original_rows, new_rows):
    if old["agent"] != new["agent"]:
        print(
            old["id"],
            old["agent"], "->", new["agent"],
            "| Expected:", new["expected"]
        )


ORIGINAL GRAPH (8 keywords)
Caught: 4
Missed: 2
Over-escalated: 3

ENHANCED GRAPH (8 keywords + context)
Caught: 6
Missed: 0
Over-escalated: 1

CHANGED DECISIONS:
A03 close -> escalate | Expected: escalate
A05 close -> escalate | Expected: escalate
A07 escalate -> close | Expected: close
A12 escalate -> close | Expected: close


In [101]:

# PART 9 - Step 4: Save experimental results

import csv

# Combine the four keyword experiments with the enhanced graph
FINAL_RESULTS = MY_RESULTS.copy()

FINAL_RESULTS.append({
    "configuration": "enhanced graph (8 words + context)",
    "caught": new_c,
    "missed": new_m,
    "over_escalated": new_o,
    "correctly_closed": 6 - new_o,
    "missed_ids": ",".join(
        r["id"] for r in new_rows
        if r["verdict"] == "MISSED"
    ),
    "over_ids": ",".join(
        r["id"] for r in new_rows
        if r["verdict"] == "over-escalated"
    )
})

with open("results_mine.csv", "w", newline="",
          encoding="utf-8") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "configuration",
            "caught",
            "missed",
            "over_escalated",
            "correctly_closed",
            "missed_ids",
            "over_ids"
        ]
    )

    writer.writeheader()
    writer.writerows(FINAL_RESULTS)

print("Saved results_mine.csv")

for r in FINAL_RESULTS:
    print(
        r["configuration"],
        "| Caught:", r["caught"],
        "| Missed:", r["missed"],
        "| Over-escalated:", r["over_escalated"]
    )


Saved results_mine.csv
narrow (2 words) | Caught: 1 | Missed: 5 | Over-escalated: 1
lecture (5 words) | Caught: 2 | Missed: 4 | Over-escalated: 3
wider (8 words) | Caught: 4 | Missed: 2 | Over-escalated: 3
widest (11 words) | Caught: 4 | Missed: 2 | Over-escalated: 5
enhanced graph (8 words + context) | Caught: 6 | Missed: 0 | Over-escalated: 1


In [102]:

# PART 9 - Step 5: Final validation

import os
import csv

assert len(MY_ALERTS) == 12
assert sum(a["expected"] == "escalate"
           for a in MY_ALERTS) == 6
assert sum(a["expected"] == "close"
           for a in MY_ALERTS) == 6

# Check the two required hard cases
baseline_words = KEYWORD_LISTS["lecture (5 words)"]

hard_missed = [
    a["id"] for a in MY_ALERTS
    if a["expected"] == "escalate"
    and not any(w in a["alert"].lower()
                for w in baseline_words)
]

hard_over = [
    a["id"] for a in MY_ALERTS
    if a["expected"] == "close"
    and any(w in a["alert"].lower()
            for w in baseline_words)
]

assert hard_missed, "Missing required hard case: missed incident"
assert hard_over, "Missing required hard case: routine keyword alert"

assert os.path.exists("results_mine.csv")

with open("results_mine.csv", newline="") as f:
    saved_rows = list(csv.DictReader(f))

assert len(saved_rows) == 5, "Expected five experiment rows"

print("Dataset validation: PASSED")
print("Missed-detection hard cases:", hard_missed)
print("Over-escalation hard cases:", hard_over)
print("CSV validation: PASSED")
print("Total experiments:", len(saved_rows))
print("Part 9 validation complete!")


Dataset validation: PASSED
Missed-detection hard cases: ['A03', 'A04', 'A05', 'A06']
Over-escalation hard cases: ['A07', 'A11', 'A12']
CSV validation: PASSED
Total experiments: 5
Part 9 validation complete!


---

## What to write up

One PDF report and a video of five to ten minutes. See `MiniProject_Agent.md` for the full
requirements. The questions that matter:

- The two errors are not equal. Which would you rather your agent made, and why? Answer as the
  person who has to staff the team.
- Which rule would you deploy, and what does it cost you? Name the alerts you are choosing to get
  wrong.
- Some alerts stay wrong at every setting. Explain why in terms of what a keyword rule can and
  cannot see, and describe the change you made to the graph and what it bought you.
- What did the graph give you that a LangChain chain could not? Point at your conditional edge and
  at `thread_id` memory, using your own output as evidence, and say when a chain would be the better
  choice.
- Twelve alerts means one alert is a sixth of a column. How much confidence does that justify, and
  what would you measure next?

Submit one PDF report with the link to your project folder and the link to your video on its first
page. The notebook, `my_alerts.json`, and your results live in the repository, which is private, with
your instructor and the TA added as collaborators. The notebook must be committed with its outputs
showing.